#A) Installs and imports

In [ ]:
!pip -q install moabb mne braindecode

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from moabb.datasets import BNCI2014_001
from braindecode.datasets import MOABBDataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[1])
print(dataset.description)

In [ ]:
raw = dataset.datasets[0].raw

In [ ]:
# Get events and event IDs from raw annotations
events, event_id = mne.events_from_annotations(raw)
print("Event IDs:", event_id)
print("First event [sample, 0, id]:", events[0])

In [ ]:
#  trial onset sample and 7.5 s window (BCI IV-2a trial length = 7.5 s)
trial_id = 0
sfreq = raw.info['sfreq']
start_sample = events[trial_id, 0]
stop_sample = start_sample + int(7.5 * sfreq)

In [ ]:
picks = ['C3', 'Cz', 'C4', 'EOG1', 'EOG2', 'EOG3']
data, times = raw.get_data(
    picks=picks,
    start=start_sample,
    stop=stop_sample,
    return_times=True
)

In [ ]:
# Shift times to start at 0 s for trial view
rel_times = times - times[0]

In [ ]:
id_to_event = {code: name for name, code in event_id.items()}
event_name = id_to_event[events[trial_id, 2]]

In [ ]:
for i, ch_name in enumerate(picks):
  plt.figure(figsize=(12, 2))
  plt.plot(rel_times, data[i], label=ch_name)
  plt.xlabel('Time relative to trial start (s)')
  plt.ylabel('Amplitude')
  plt.title(f'Subject 1 - {trial_id + 1} Trial {event_name}')
  plt.legend()

plt.tight_layout()
plt.show()

##Visualize 3d channel pos

In [ ]:
montage = raw.get_montage()
print("Montage:", montage)

if montage is not None:
    pos_dict = montage.get_positions()['ch_pos']  # {ch_name: array([x, y, z])}
    ch_names = [ch for ch in ['C3', 'Cz', 'C4'] if ch in pos_dict]
    coords = np.array([pos_dict[ch] for ch in ch_names])  # shape (n_ch, 3)

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(coords[:, 0]*100, coords[:, 1]*100, coords[:, 2]*100, s=80, c='red')
    for ch, (x, y, z) in zip(ch_names, coords):
        ax.text(x*100, y*100, z*100, f' {ch}')
    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    ax.set_zlabel('Z (cm)')
    ax.set_title('3D Electrode Positions')
    plt.show()